In [1]:
import os
os.chdir("../..")
from src.integration.query_ontology import (build_template_from_regions, save_json,
                                            build_dict, json_to_csv, read_json)
import pprint
from src.common.utils import init_ont

import subprocess
import ast
from src.integration.SentenceEmbedder import SentenceEmbedder, OntologyEmbedder
import yaml

/home/walter/projects/brain_tumor_detection/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import lovely_tensors as lt
lt.monkey_patch()

In [3]:
with open("config/keys.yaml", "r") as f:
    keys = yaml.safe_load(f)

ont_key = keys["ont_key"]
BIOPORTAL_KEY = keys["bio_portal_key"]

In [4]:
init_ont(key=ont_key)

[!!] Initializing integration module...
[OK] Initialization complete.


In [5]:
with open("config/parameters.yaml", "r") as f:
    config = yaml.safe_load(f)

json_regions = config["json_regions"]
csv_regions = config["csv_regions"]
embeddings_path = config["embeddings_path"]

In [6]:
json_regions

'src/integration/txt/solved_regions.json'

In [3]:
'''
def fetch_definition_from_bioportal(uberon_id):

    iri = f"http://purl.obolibrary.org/obo/{uberon_id.replace(':','_')}"
    encoded_iri = quote(iri, safe='')
    url = f"https://data.bioontology.org/ontologies/UBERON/classes/{encoded_iri}"
    headers = {"Authorization": f"apikey token={BIOPORTAL_KEY}"}

    try:
        resp = requests.get(url, headers=headers, timeout=10)
        if resp.status_code == 200:
            data = resp.json()
            defs = data.get("definition", [])
            if not defs:
                defs = data.get("comment", [])
            return defs
        else:
            print(f"[BioPortal] Error {resp.status_code} for {uberon_id}")
    except Exception as e:
        print(f"[BioPortal] Exception for {uberon_id}: {e}")
    return []

def build_dict(brain_dict, term, t_value=10):

    result = subprocess.run(
        ["scig", "-t", str(t_value), "t", term, "-v"],
        capture_output=True,
        text=True
    )
    output_str = result.stdout
    lines = output_str.splitlines()
    if not lines:
        return brain_dict

    zone = lines[0].strip()
    brain_dict[zone] = {}

    kv_pattern = re.compile(r'\s*(\w+):\s*(.*)')
    current_ontology = None

    for line in lines[1:]:
        line = line.rstrip()
        if not line.strip():
            continue

        if re.match(r'^\s*[A-Z]+[:_]\d+', line):
            current_ontology = line.strip()
            if current_ontology not in brain_dict[zone]:
                brain_dict[zone][current_ontology] = {}
            continue

        if current_ontology is not None:
            match = kv_pattern.match(line)
            if match:
                key, value = match.groups()
                if value.startswith("[") and value.endswith("]"):
                    try:
                        value = ast.literal_eval(value)
                    except:
                        pass
                if value in ["True", "False"]:
                    value = (value == "True")
                brain_dict[zone][current_ontology][key] = value

    for ont in brain_dict[zone]:
        if ont.startswith("UBERON:"):
            defs = fetch_definition_from_bioportal(ont)
            brain_dict[zone][ont]['definitions'] = defs

    return brain_dict

def json_to_csv(data, output_path):
    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["label", "query", "ontology_id", "definition"])

        for label, content in data.items():
            query = content.get("query", "")
            definitions = content.get("definitions", {})

            for ont_id, definition in definitions.items():
                writer.writerow([
                    label,
                    query,
                    ont_id,
                    definition
                ])

def save_json(data, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=5, ensure_ascii=False)
'''

<>:1: SyntaxWarning: invalid escape sequence '\s'
<>:1: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_385282/306684417.py:1: SyntaxWarning: invalid escape sequence '\s'
  '''


'\ndef fetch_definition_from_bioportal(uberon_id):\n\n    iri = f"http://purl.obolibrary.org/obo/{uberon_id.replace(\':\',\'_\')}"\n    encoded_iri = quote(iri, safe=\'\')\n    url = f"https://data.bioontology.org/ontologies/UBERON/classes/{encoded_iri}"\n    headers = {"Authorization": f"apikey token={BIOPORTAL_KEY}"}\n\n    try:\n        resp = requests.get(url, headers=headers, timeout=10)\n        if resp.status_code == 200:\n            data = resp.json()\n            defs = data.get("definition", [])\n            if not defs:\n                defs = data.get("comment", [])\n            return defs\n        else:\n            print(f"[BioPortal] Error {resp.status_code} for {uberon_id}")\n    except Exception as e:\n        print(f"[BioPortal] Exception for {uberon_id}: {e}")\n    return []\n\ndef build_dict(brain_dict, term, t_value=10):\n\n    result = subprocess.run(\n        ["scig", "-t", str(t_value), "t", term, "-v"],\n        capture_output=True,\n        text=True\n    )\

In [3]:
def fetch_edges(term):

    result = subprocess.run(
        f"scig g {term} --edges --verbose",
        shell=True,
        capture_output=True,
        text=True
    )

    edges = []

    for line in result.stdout.splitlines():
        line = line.strip()
        if not line:
            continue

        if line.startswith("(") and ") ;" in line:
            triple_part, meta_part = line.split(") ;", 1)
            triple_part = triple_part[1:]  # remove '('
            parts = triple_part.split()

            if len(parts) != 3:
                continue

            predicate, subject, obj = parts

            try:
                meta = ast.literal_eval(meta_part.strip())
                owl_type = meta.get("owlType", [None])[0]
            except Exception:
                owl_type = None

            edges.append({
                "subject": subject,
                "predicate": predicate,
                "object": obj,
                "owlType": owl_type
            })

    return edges

from collections import defaultdict

def edges_to_dict(edges):
    graph = defaultdict(list)

    for e in edges:
        s = e["subject"]
        graph[s].append({
            "predicate": e["predicate"],
            "object": e["object"],
            "owlType": e.get("owlType")
        })

    return dict(graph)

from collections import defaultdict

def print_all_edges(node, edges_list):
    """
    Print all edges where `node` is subject or object.
    
    edges_list: list of dicts returned by fetch_edges()
    """
    graph = defaultdict(list)
    rev_graph = defaultdict(list)

    for e in edges_list:
        s = e["subject"]
        o = e["object"]
        p = e["predicate"]
        owl = e.get("owlType")
        
        graph[s].append({"predicate": p, "object": o, "owlType": owl})
        rev_graph[o].append({"predicate": p, "subject": s, "owlType": owl})

    for e in graph.get(node, []):
        print(f"{node} --{e['predicate']}--> {e['object']} (owlType={e['owlType']})")

    for e in rev_graph.get(node, []):
        print(f"{e['subject']} --{e['predicate']}--> {node} (owlType={e['owlType']})")
        
        import subprocess

def fetch_relation_label(rel_id):
    """
    Fetch human-readable label for a relationship ID from SciGraph
    using subprocess and `scig t <rel_id> --verbose`.
    Returns the label as string, or fallback to the ID if not found.
    """
    result = subprocess.run(
        f"scig t {rel_id} --verbose",
        shell=True,
        capture_output=True,
        text=True
    )
    
    output_lines = result.stdout.splitlines()
    
    # Try to find a line with the label
    # SciGraph usually prints the ID on the first line
    # and label in subsequent lines, e.g., 'label: part of'
    for line in output_lines:
        line = line.strip()
        if line.lower().startswith("label:") or line.lower().startswith("name:"):
            # extract after 'label: '
            return line.split(":", 1)[1].strip()
    
    # fallback to the ID
    return rel_id

In [5]:
# Example usage with your edges
edges_list = fetch_edges("UBERON:0000411")

for e in edges_list:
    s = e["subject"]
    p = e["predicate"]
    o = e["object"]
    owl = e.get("owlType")
    
    readable_pred = fetch_relation_label(p)
    print(f"{s} --{readable_pred}--> {o} (owlType={owl})")

UBERON:8440055 --BFO:0000050--> UBERON:0000411 (owlType=subClassOf)
UBERON:8440059 --BFO:0000050--> UBERON:0000411 (owlType=subClassOf)
UBERON:8440054 --BFO:0000050--> UBERON:0000411 (owlType=subClassOf)
UBERON:0000411 --BFO:0000050--> UBERON:0000411 (owlType=subClassOf)
UBERON:8440056 --BFO:0000050--> UBERON:0000411 (owlType=subClassOf)
UBERON:0034750 --BFO:0000050--> UBERON:0000411 (owlType=subClassOf)
UBERON:0022232 --BFO:0000050--> UBERON:0000411 (owlType=subClassOf)
UBERON:0014370 --BFO:0000050--> UBERON:0000411 (owlType=subClassOf)
UBERON:8440010 --BFO:0000050--> UBERON:0000411 (owlType=subClassOf)
UBERON:0035897 --BFO:0000050--> UBERON:0000411 (owlType=subClassOf)
UBERON:0002436 --BFO:0000050--> UBERON:0000411 (owlType=subClassOf)
UBERON:0035900 --BFO:0000050--> UBERON:0000411 (owlType=subClassOf)
UBERON:8440050 --BFO:0000050--> UBERON:0000411 (owlType=subClassOf)
UBERON:0000411 --BFO:0000050--> UBERON:0002021 (owlType=subClassOf)
UBERON:8440052 --BFO:0000050--> UBERON:0000411 (

In [6]:
import requests
import urllib.parse

def resolve_property_ols_json(curie):
    """
    Resolve a property CURIE (like BFO:0000050) to its label and definition using OLS JSON API.
    """
    prefix, local_id = curie.split(":")
    iri = f"http://purl.obolibrary.org/obo/{prefix}_{local_id}"
    encoded_iri = urllib.parse.quote(iri, safe='')

    # Endpoint corretto con query parameter
    url = f"https://www.ebi.ac.uk/ols4/api/ontologies/ro/properties?iri={encoded_iri}"

    headers = {"Accept": "application/json"}
    response = requests.get(url, headers=headers)

    if response.status_code != 200:
        print("Error:", response.status_code)
        return None

    data = response.json()
    # OLS restituisce risultati in "_embedded" -> "properties"
    props = data.get("_embedded", {}).get("properties", [])
    if not props:
        return None

    prop = props[0]
    label = prop.get("label")
    definition = prop.get("definition", [None])[0] if prop.get("definition") else None

    return {"label": label, "definition": definition}

In [7]:
info = resolve_property_ols_json("RO:0002175")
print("Label:", info["label"] if info else None)
print("Definition:", info["definition"] if info else None)

Label: present in taxon
Definition: None


In [7]:
brain_dict = {}
brain_dict = build_dict(brain_dict, "nucleus amygdaloideus centralis", t_value=10)
pprint.pprint(brain_dict)

{'nucleus amygdaloideus centralis': {'FMA:74047': {'abbreviations': [],
                                                   'acronyms': [],
                                                   'categories': [],
                                                   'definitions': [],
                                                   'deprecated': False,
                                                   'iri': 'http://purl.org/sig/ont/fma/fma74047',
                                                   'labels': ['Central '
                                                              'amygdaloid '
                                                              'nucleus'],
                                                   'synonyms': ['Nucleus '
                                                                'amygdaloideus '
                                                                'centralis',
                                                                'Noyau central '
               

In [9]:
regions_mapping = {
    "GM Amygdala_centromedial group" : "nucleus amygdaloideus centralis",
    "GM Amygdala_laterobasal group" : "basolateral nucleus of amygdala",
    "GM Amygdala_superficial group" : "nucleus corticalis amygdalae",
    "GM Anterior intra-parietal sulcus hIP1" : "intraparietal sulcus",
    "GM Anterior intra-parietal sulcus hIP2" : "intraparietal sulcus",
    "GM Anterior intra-parietal sulcus hIP3" : "intraparietal sulcus",
    "GM Broca's area BA44" : "BA44",
    "GM Broca's area BA45" : "BA45",
    "GM Hippocampus cornu ammonis" : "cornu ammonis",
    "GM Hippocampus dentate gyrus" : "dentate gyrus",
    "GM Hippocampus entorhinal cortex" : "entorhinal cortex",
    "GM Hippocampus hippocampal-amygdaloid transition area" : "claustral amygdaloid area",
    "GM Hippocampus subiculum" : "subiculum",
    "GM Inferior parietal lobule PF" : "inferior parietal lobule",
    "GM Inferior parietal lobule PFcm": "inferior parietal lobule",
    "GM Inferior parietal lobule PFm" : "inferior parietal lobule",
    "GM Inferior parietal lobule PFop" : "inferior parietal lobule",
    "GM Inferior parietal lobule PFt" : "inferior parietal lobule",
    "GM Inferior parietal lobule PGp" : "inferior parietal lobule",
    "GM Inferior parietal lobule Pga" : "inferior parietal lobule",
    "GM Insula Id1" : "insula",
    "GM Insula Ig1" : "insula",
    "GM Insula Ig2" : "insula",
    "GM Lateral geniculate body" : "lateral geniculate body",
    "GM Mamillary body" : "mammillary body",
    "GM Medial geniculate body" : "medial geniculate body",
    "GM Premotor cortex BA6" : "premotor cortex",
    "GM Primary auditory cortex TE1.0" : "primary auditory cortex",
    "GM Primary auditory cortex TE1.1" : "primary auditory cortex",
    "GM Primary auditory cortex TE1.2" : "primary auditory cortex",
    "GM Primary motor cortex BA4a" : "primary motor cortex",
    "GM Primary motor cortex BA4p" : "primary motor cortex",
    "GM Primary somatosensory cortex BA1" : "primary somatosensory cortex",   
    "GM Primary somatosensory cortex BA2" : "primary somatosensory cortex",
    "GM Primary somatosensory cortex BA3a" : "primary somatosensory cortex",
    "GM Primary somatosensory cortex BA3b" : "primary somatosensory cortex",
    "GM Secondary somatosensory cortex / Parietal operculum OP1" : "secondary somatosensory cortex",
    "GM Secondary somatosensory cortex / Parietal operculum OP2" : "secondary somatosensory cortex",
    "GM Secondary somatosensory cortex / Parietal operculum OP3" : "secondary somatosensory cortex",
    "GM Secondary somatosensory cortex / Parietal operculum OP4" : "secondary somatosensory cortex",
    "GM Superior parietal lobule 5Ci" : "superior parietal lobule",
    "GM Superior parietal lobule 5L" : "superior parietal lobule",
    "GM Superior parietal lobule 5M" : "superior parietal lobule",
    "GM Superior parietal lobule 7A" : "superior parietal lobule",
    "GM Superior parietal lobule 7M" : "superior parietal lobule",
    "GM Superior parietal lobule 7P" : "superior parietal lobule",
    "GM Superior parietal lobule 7PC" : "superior parietal lobule",
    "GM Visual cortex V2 BA18": "visual cortex",
    "GM Visual cortex V3V": "visual cortex",
    "GM Visual cortex V4": "visual cortex",
    "GM Visual cortex V5": "visual cortex",
    "WM Acoustic radiation" : "auditory radiation",
    "WM Callosal body" : "corpus callosum",
    "WM Cingulum" : "cingulum of brain",
    "WM Corticospinal tract" : "corticospinal tract",
    "WM Fornix" : "fornix of brain",
    "WM Inferior occipito-frontal fascicle" : "inferior frontal sulcus",
    "WM Optic radiation" : "optic radiation",
    "WM Superior longitudinal fascicle" : "superior longitudinal fasciculus",
    "WM Superior occipito-frontal fascicle" : "superior frontal sulcus",
    "WM Uncinate fascicle" : "uncinate fasciculus"
    }

brain_dict = {}
template, brain_dict = build_template_from_regions(regions_mapping, build_dict, t_value=10)

In [21]:
save_json(template, json_regions)

In [22]:
json_to_csv(template, csv_regions)

In [ ]:
import json

with open(json_regions, "r", encoding="utf-8") as f:
    solved_regions = json.load(f)

pprint.pprint(solved_regions)

In [ ]:
for key in solved_regions:
    print("*" * 60)
    print(key)
    for item_ont in solved_regions[key]["definitions"]:
        print(f"item ID: {item_ont}")
        # Example usage with your edges
        edges_list = fetch_edges(item_ont)

        for e in edges_list:
            s = e["subject"]
            p = e["predicate"]
            o = e["object"]
            owl = e.get("owlType")
            
            readable_pred = fetch_relation_label(p)
            print(f"{s} --{readable_pred}--> {o} (owlType={owl})")
            
    print("*" * 60)
    print()

## Text Encoding

Now, we'll use an encoder to encode the text extracted from ontologies.

In [10]:
model_name = "NeuML/pubmedbert-base-embeddings"
embedder = SentenceEmbedder(model_name, device="cuda:1")

ontology_embedder = OntologyEmbedder(
    embedder=embedder,
    model_name=model_name,
    embeddings_path=embeddings_path,
    ont_name="UBERON",
    batch_size=16
)

uberon_embeddings = ontology_embedder.process(json_regions)

Using specified device: cuda:1
Using device: cuda:1


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1468.56it/s, Materializing param=pooler.dense.weight]                               


Saved to: src/integration/txt/uberon_embeddings_NeuML_pubmedbert-base-embeddings
